# Daedalus Devs Team Report # 1 (03/24/2026)

### What we have done this week:
* Installed libraries and dependencies needed **(cell 1)**
* Figured out how to upload resume PDF file to Colab and use it with the LLM
* Created a function that takes in user's resume and parse it to output the extracted info **(cell 3)**
* Created a function that uses Skyvern, played around a little to see how it works. **(cell 4)**

### Plans for next week:
* Need to figure out how to see the filled in website, not as a screenshot but in another tab.
* Follow our workflow where the user can make corrections to the extracted resume info.

In [ ]:

%pip install chromadb
# %pip install "skyvern==1.0.24"
%pip install skyvern
# %playwright install --with-deps chromium
# %playwright install chromium
%pip install langchain langchain-huggingface
%pip install python-dotenv

print ("Done installing")

In [1]:
# import os
# from dotenv import load_dotenv


# load_dotenv()


# gemini_key = os.getenv("GEMINI_API_KEY")
# skyvern_key = os.getenv("SKYVERN_API_KEY")
# phil_key = os.getenv("PHIL-KEY")


# os.environ["ENABLE_AUTH"] = "false"

# # from skyvern.forge.sdk import SkyvernForge
# from skyvern import Skyvern
# import nest_asyncio
# import asyncio

# # Apply nest_asyncio to allow nested event loops in Colab
# nest_asyncio.apply()

# # 2. Map it to the environment variables Skyvern/LiteLLM looks for
# os.environ["GEMINI_API_KEY"] = gemini_key
# # os.environ["GOOGLE_API_KEY"] = gemini_key  # Some versions prefer this
# # Optional: Force Skyvern to use a specific model
# os.environ["LLM_KEY"] = "gemini/gemini-1.5-flash"
# print(f"Key check: {'FOUND' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")
# # print("Keys loaded and environment configured.")


import os
from dotenv import load_dotenv
os.chdir("/Users/sanilachowdhury/Desktop/ai_agents")

# load .env and show where it's looking
load_dotenv()
print(f"Looking for .env in: {os.getcwd()}")
print(f"GEMINI_API_KEY found: {bool(os.getenv('GEMINI_API_KEY'))}")

gemini_key = os.getenv("GEMINI_API_KEY")
skyvern_key = os.getenv("SKYVERN_API_KEY")
phil_key = os.getenv("PHIL-KEY")

if not gemini_key:
    raise ValueError("GEMINI_API_KEY not found! Check that .env is in: " + os.getcwd())

os.environ["ENABLE_AUTH"] = "false"

from skyvern import Skyvern
import nest_asyncio
import asyncio

nest_asyncio.apply()

os.environ["GEMINI_API_KEY"] = gemini_key
os.environ["LLM_KEY"] = "gemini/gemini-2.0-flash-lite"
print(f"Key check: {'FOUND' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")

Looking for .env in: /Users/sanilachowdhury/Desktop/ai_agents
GEMINI_API_KEY found: True
Key check: FOUND


In [ ]:
url = "https://demo.applitools.com/"

In [6]:
# !skyvern quickstart
# from skyvern import Skyvern
# import asyncio

# async def run_locally():
#   client = Skyvern.local()
#   browser = await client.launch_local_browser()
#   page = await browser.get_working_page()
#   await page.goto(url)
#   await page.act("My username is user101 and my password is hello123. Please fill in those fields but DO NOT press submit/login")
#   print("page filled in!\n")

#   await browser.close()

# await run_locally()

from skyvern import Skyvern
import asyncio
import nest_asyncio
from google import genai
# from google.colab import userdata

nest_asyncio.apply()

client_genai = genai.Client(api_key=gemini_key)

# this takes in user's resume and parse and LLM will output the info it extracted to ensure accuracy
def parse_pdf_with_gemini(file_path, prompt="Please parse this resume and return its contents in a structured JSON that is easy for both LLM and humans to read."):
    """
    Uploads a PDF to Gemini and returns the model's analysis.
    """
    # upload user's resume
    print(f"Uploading {file_path}...")
    uploaded_file = client_genai.files.upload(
        file=file_path,
        config={"display_name": "User_PDF"}
    )

   # feeds LLM the prompt and uploaded resume PDF
    response = client_genai.models.generate_content(
        # model="gemini-2.0-flash",
        model="gemini-2.5-flash-lite",
        contents=[uploaded_file, prompt]
    )

    return response.text

# check if agent extracted the resume's info correctly
def verify_resume_info(extracted_info):
    """
    Shows the extracted info to the user and allows them to correct it.
    Keeps looping until the user confirms it's correct.
    """
    current_info = extracted_info

    while True:
        print("\n" + "="*10)
        print("EXTRACTED RESUME INFO:")
        print("="*10)
        print(current_info)
        print("="*10)

        user_confirm = input("\nIs this information correct? (y/n): ").strip().lower()

        if user_confirm == "y":
            print("\nGreat! Moving forward with this information.")
            return current_info
        else:
            correction = input("\nWhat needs to be corrected or added? Please describe: ").strip()

            print("\nUpdating extracted info...")
            response = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"""Here is the currently extracted resume information:
{current_info}

The user wants the following correction or addition:
{correction}

Please update the extracted resume JSON accordingly and return the full updated JSON."""
                ]
            )
            current_info = response.text
            print("\nInfo updated! Let's review it again.")


# call the function to store the extracted info
result = parse_pdf_with_gemini("content/jakes-resume.pdf")

# run verify_resume_info to verify and correct
final_info = verify_resume_info(result)

print("\nFinal verified resume info:")
print(final_info)
# call the function to store the extracted info and print it
# result = parse_pdf_with_gemini("content/jakes-resume.pdf")
# print(result)




Uploading content/jakes-resume.pdf...

EXTRACTED RESUME INFO:
```json
{
  "name": "Jake Ryan",
  "contact": {
    "phone": "123-456-7890",
    "email": "jake@su.edu",
    "linkedin": "linkedin.com/in/jake",
    "github": "github.com/jake"
  },
  "education": [
    {
      "institution": "Southwestern University",
      "degree": "Bachelor of Arts in Computer Science, Minor in Business",
      "location": "Georgetown, TX",
      "startDate": "Aug. 2018",
      "endDate": "May 2021"
    },
    {
      "institution": "Blinn College",
      "degree": "Associate's in Liberal Arts",
      "location": "Bryan, TX",
      "startDate": "Aug. 2014",
      "endDate": "May 2018"
    }
  ],
  "experience": [
    {
      "title": "Undergraduate Research Assistant",
      "institution": "Texas A&M University",
      "location": "College Station, TX",
      "startDate": "June 2020",
      "endDate": "Present",
      "description": [
        "Developed a REST API using FastAPI and PostgreSQL to store da

In [ ]:

from skyvern import Skyvern
import asyncio
import nest_asyncio

nest_asyncio.apply()


# testing Skyvern to see how it would fill in the website with the info given
async def run_with_cloud():
    client = Skyvern(api_key=skyvern_key)

    result = await client.run_task(
        url="https://demo.applitools.com/",
        prompt="My username is user101 and my password is hello123. Please fill in those fields but DO NOT press submit/login."
    )

    run_id = result.run_id
    print(f"Task started: {run_id}")

    # print status while Skyvern runs
    while True:
        run = await client.get_run(run_id)
        print(f"Status: {run.status}")
        if run.status in ["completed", "failed", "terminated", "timed_out", "canceled"]:
            break
        await asyncio.sleep(5)

    print(f"Final status: {run.status}")
    print(f"Output: {run.output}")

await run_with_cloud()

In [ ]:
%pip uninstall -y cffi
%pip install cffi

In [32]:
import os
os.environ["ENABLE_AUTH"] = "false"

import subprocess
import time

# start Skyvern server in the background
server = subprocess.Popen(
    ["skyvern", "run", "server"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# wait for server to start
print("Starting Skyvern server...")
time.sleep(10)
print("Server should be running on localhost:8000")

Starting Skyvern server...
Server should be running on localhost:8000


In [31]:
server.terminate()
print("Server stopped.")

Server stopped.


In [21]:
print([m for m in dir(Skyvern) if not m.startswith("_")])

['aclose', 'agent', 'artifacts', 'cancel_run', 'change_tier_api_v1billing_change_tier_post', 'close_browser_session', 'connect_to_browser_over_cdp', 'connect_to_cloud_browser_session', 'create_browser_profile', 'create_browser_session', 'create_checkout_session_api_v1billing_checkout_post', 'create_credential', 'create_folder', 'create_portal_session_api_v1billing_portal_post', 'create_script', 'create_workflow', 'delete_browser_profile', 'delete_credential', 'delete_folder', 'delete_workflow', 'deploy_script', 'download_files', 'environment', 'get_artifact', 'get_browser_profile', 'get_browser_session', 'get_browser_sessions', 'get_credential', 'get_credentials', 'get_folder', 'get_folders', 'get_organization_billing_api_v1billing_state_get', 'get_run', 'get_run_artifacts', 'get_run_timeline', 'get_script', 'get_scripts', 'get_workflow', 'get_workflow_runs', 'get_workflow_versions', 'get_workflows', 'launch_cloud_browser', 'launch_local_browser', 'list_browser_profiles', 'local', 'log

In [27]:
import inspect
print(inspect.signature(Skyvern.connect_to_browser_over_cdp))

(self, cdp_url: str) -> skyvern.library.skyvern_browser.SkyvernBrowser


In [33]:
from skyvern import Skyvern
# os.environ["ENABLE_AUTH"] = "false"
# !python3 -m playwright install chromium

# try:
#     # import _cffi_backend
#     import OpenSSL
#     print("✅ Success! CFFI backend found.")
# except ImportError:
#     print("❌ Still missing. Check your terminal output for errors during installation.")

# async def run_locally():
#     client = Skyvern.local()
    
#     browser_args = [
#         "--disable-gpu",
#         "--disable-software-rasterizer",
#         "--no-sandbox"
#     ]
#     browser = await client.launch_local_browser(headless=False, args=browser_args)  # Make browser visible
#     page = await browser.get_working_page()
#     await page.goto("https://demo.applitools.com/")
#     print("Done opening page!\n")
#     await page.act("My username is user101 and my password is hello123. Please fill in those fields but DO NOT press submit/login")
#     print("Page filled in!\n")
#     # Note: Browser will remain open; close manually if needed
#     # await browser.close()  # Uncomment to close after

# await run_locally()

async def run_locally():
    client = Skyvern.local()
    
    browser = await client.connect_to_browser_over_cdp(cdp_url="http://127.0.0.1:9222")
    page = await browser.get_working_page()
    await page.goto("https://demo.applitools.com/")
    print("Done opening page!\n")
    await page.act("My username is user101 and my password is hello123. Please fill in those fields but DO NOT press submit/login")
    print("Page filled in!\n")

await run_locally()

2026-03-28T23:38:57.278999Z [info     ] [skyvern_browser_page_ai.py    :214 ] AI act                         [skyvern.library.skyvern_browser_page_ai] prompt=My username is user101 and my password is hello123. Please fill in those fields but DO NOT press submit/login workflow_run_id=None entrypoint=ipykernel_launcher


Done opening page!



/Users/sanilachowdhury/Desktop/ai_agents/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:365: InsecureKeyLengthWarning: The HMAC key is 11 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  decoded = self.decode_complete(
/Users/sanilachowdhury/Desktop/ai_agents/.venv/lib/python3.13/site-packages/skyvern/forge/sdk/forge_log.py:162: UserWarning: Remove `format_exc_info` from your processor chain if you want pretty exceptions.
  rendered = super().__call__(logger, name, event_dict)
2026-03-28T23:38:57.566454Z [error    ] [org_auth_service.py           :203 ] Error decoding JWT             [skyvern.forge.sdk.services.org_auth_service] entrypoint=ipykernel_launcher error_type=jwt.exceptions.InvalidSignatureError error_category=ERROR exception_hash=395ffdf1072f4c75
Traceback (most recent call last):
  File "/Users/sanilachowdhury/Desktop/ai_agents/.venv/lib/python3.13/site-packages/anyio/_backends/_asyncio.py", line 803, in __aexi

ApiError: headers: {'content-length': '43', 'content-type': 'application/json'}, status_code: 403, body: {'detail': 'Could not validate credentials'}

In [ ]:
import os
from dotenv import load_dotenv
from skyvern import Skyvern
import nest_asyncio

nest_asyncio.apply()
load_dotenv()

async def run_locally():
    client = Skyvern.local()
    
    browser = await client.connect_to_browser_over_cdp(
        browser_address="http://127.0.0.1:9222"
    )
    page = await browser.get_working_page()
    await page.goto("https://demo.applitools.com/")
    print("Done opening page!\n")
    await page.act("My username is user101 and my password is hello123. Please fill in those fields but DO NOT press submit/login")
    print("Page filled in!\n")

await run_locally()

In [ ]:
import os
import subprocess
from playwright.async_api import async_playwright

# %playwright install
# 1. Corrected path check (os.path.expanduser)
path = os.path.expanduser("~/.cache/ms-playwright/chromium-1129/chrome-linux/chrome")

print(f"Checking path: {path}")

if os.path.exists(path):
    print("✅ Chromium executable found!")
else:
    print("❌ Still missing. Please run 'python3 -m playwright install chromium' in your terminal.")

# 2. Check if we can actually launch it
try:
    async def test_launch():
        async with async_playwright() as p:
            # We'll test with headless=True first to verify the engine
            browser = await p.chromium.launch(headless=True)
            print("✅ Playwright Engine is working!")
            await browser.close()
    
    import asyncio
    await test_launch()
except Exception as e:
    print(f"❌ Launch failed: {e}")

# Plans for next week:
- Need to figure out how to see the filled in website, not as a screenshot but in another tab.
- Follow our workflow where the user can make corrections to the extracted resume info.
- put agent graph at the top of notebook for next week